In [0]:
-- DATA QUALITY AND VALIDATION REPORT

-- Inspect inferred data types and nullability for the raw table
DESCRIBE EXTENDED workspace.d3_raw.customer_raw;


-- Check for missing or duplicate CustomerIds
SELECT 
    COUNT(*) AS total_records,
    COUNT(CustomerId) AS non_null_ids,
    COUNT(DISTINCT CustomerId) AS unique_ids,
    COUNT(*) - COUNT(DISTINCT CustomerId) AS duplicate_id_count
FROM workspace.d3_raw.customer_raw;


-- Identify exact duplicate customer entries based on email or identity
SELECT Email, COUNT(*) AS duplicate_count
FROM workspace.d3_raw.customer_raw
GROUP BY Email
HAVING COUNT(*) > 1;


-- Quantify completeness across core text and contact columns
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN FirstName IS NULL THEN 1 ELSE 0 END) AS null_first_names,
    SUM(CASE WHEN LastName IS NULL THEN 1 ELSE 0 END) AS null_last_names,
    SUM(CASE WHEN Email IS NULL THEN 1 ELSE 0 END) AS null_emails,
    SUM(CASE WHEN State IS NULL THEN 1 ELSE 0 END) AS null_states,
    SUM(CASE WHEN Company IS NULL THEN 1 ELSE 0 END) AS null_companies,
    SUM(CASE WHEN Phone IS NULL THEN 1 ELSE 0 END) AS null_phones,
    SUM(CASE WHEN Fax IS NULL THEN 1 ELSE 0 END) AS null_faxes
FROM workspace.d3_raw.customer_raw;


-- Detect whitespace issues, casing inconsistencies, or malformed emails
SELECT 
    CustomerId,
    FirstName,
    LastName,
    Email,
    PostalCode
FROM workspace.d3_raw.customer_raw
WHERE 
    -- Leading/trailing whitespaces
    FirstName LIKE ' %' OR FirstName LIKE '% '
    OR LastName LIKE ' %' OR LastName LIKE '% '
    -- Email missing standard domain structure
    OR Email NOT LIKE '%@%.%'
    -- State lowercase or inconsistent
    OR (State IS NOT NULL AND State != UPPER(State));


-- Create one reporting-ready row per customer in dim_customer

CREATE OR REPLACE TABLE workspace.d3_mart.dim_customer AS
SELECT
    -- Business Keys & Attributes
    customer_id,
    first_name,
    last_name,
    full_name,
    -- company_name,
    email_address,
    phone_number,
    address,
    city,
    -- state,
    country,
    postal_code,
    support_rep_id,
    --faxes
    
    -- Mart Audit Column
    CURRENT_TIMESTAMP() AS mart_created_at
FROM workspace.d3_clean.customer_clean;

SELECT * FROM workspace.d3_mart.dim_customer;

